In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["DISABLE_TF_IMPORT"] = "1"


In [3]:
pip install -U transformers accelerate peft bitsandbytes


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
#pip install openpyxl

In [5]:
pip install peft

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from torch.utils.data import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import BitsAndBytesConfig


from peft import LoraConfig, get_peft_model


In [7]:
df = pd.read_excel("Reddit_Combined_Data.xlsx")

def combine_text(row):
    title = str(row.get("title", "") or "")
    body = str(row.get("selftext", "") or "")
    return title.strip() + " [SEP] " + body.strip()

df["text"] = df.apply(combine_text, axis=1)
print("Example combined text:")
print(df[["title", "selftext", "text"]].head(3))
# Always work on text + labels only
df["text"] = df["text"].astype(str).str.strip()

print("Dataset shape:", df.shape)


Example combined text:
                                               title  \
0                        Do people get over anxiety?   
1  does anyone else have this big fear of suddenl...   
2         3 hour long panic attack after trying weed   

                                            selftext  \
0  Tried to watch this documentary “anxious Ameri...   
1  i’m currently laying in bed wide awake, feelin...   
2  Second time trying weed. First time felt close...   

                                                text  
0  Do people get over anxiety? [SEP] Tried to wat...  
1  does anyone else have this big fear of suddenl...  
2  3 hour long panic attack after trying weed [SE...  
Dataset shape: (800, 13)


In [8]:
# ============================================================
# 4. Basic cleaning for key raw columns
# ============================================================
def clean_str(x):
    if x is None:
        return np.nan
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null", "", "na"]:
        return np.nan
    return x

for col in ["Perceived severity", "Self-efficacy", "Cue to Action", "Sentiment"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_str)


In [9]:
# ============================================================
# 5. PERCEIVED SEVERITY  → severity_merged
# ============================================================
def map_severity(raw):
    if pd.isna(raw):
        return "Not mentioned"

    s = str(raw).lower()

    if "anxiety" in s or "panic" in s:
        return "Anxiety/Panic"
    if "depress" in s:
        return "Depression"
    if "ptsd" in s or "trauma" in s or "severe" in s:
        return "Severe/PTSD"
    if "lonely" in s or "isolation" in s or "alone" in s:
        return "Loneliness/Isolation"
    if "addict" in s or "alcohol" in s or "drug" in s or "substance" in s:
        return "Substance-related"

    return "Other"

df["severity_merged"] = df["Perceived severity"].apply(map_severity)
print("severity_merged distribution:")
print(df["severity_merged"].value_counts(dropna=False))

severity_merged distribution:
severity_merged
Depression           343
Anxiety/Panic        277
Other                138
Substance-related     23
Severe/PTSD           19
Name: count, dtype: int64


In [10]:
# ============================================================
# 6. SELF-EFFICACY (binary merged)  → Self_eff_merged
# ============================================================
def map_self_efficacy_binary(raw):
    if pd.isna(raw):
        return np.nan

    s = str(raw).strip()

    if s in ["Empowered Mindset", "Overcome Mindset"]:
        return "High self-efficacy"
    if s in ["Troubled Mindset", "Denial Mindset"]:
        return "Low self-efficacy"

    return np.nan

df["Self_eff_merged"] = df["Self-efficacy"].apply(map_self_efficacy_binary)
print("Self_eff_merged distribution:")
print(df["Self_eff_merged"].value_counts(dropna=False))

Self_eff_merged distribution:
Self_eff_merged
Low self-efficacy     519
High self-efficacy    281
Name: count, dtype: int64


In [11]:
# ============================================================
# 7. SELF-EFFICACY (4-way UNGROUPED) → df_selfeff4
# ============================================================
valid_self_eff_4 = [
    "Troubled Mindset",
    "Overcome Mindset",
    "Empowered Mindset",
    "Denial Mindset",
]

df_selfeff4 = df[df["Self-efficacy"].isin(valid_self_eff_4)].copy()
df_selfeff4["text_len"] = df_selfeff4["text"].astype(str).str.len()
df_selfeff4 = df_selfeff4[df_selfeff4["text_len"] >= 50].copy()

print("Self-efficacy 4-way distribution (filtered):")
print(df_selfeff4["Self-efficacy"].value_counts(dropna=False))
print("Rows in 4-way self-efficacy:", len(df_selfeff4))

Self-efficacy 4-way distribution (filtered):
Self-efficacy
Troubled Mindset     492
Overcome Mindset     150
Empowered Mindset    130
Denial Mindset        27
Name: count, dtype: int64
Rows in 4-way self-efficacy: 799


In [12]:
# ============================================================
# 8. CUE TO ACTION  → Cue_group
# ============================================================
def map_cue_action(raw):
    if pd.isna(raw):
        return "No clear action"

    s = str(raw).lower()

    if "help" in s or "therapy" in s or "treatment" in s or "doctor" in s:
        return "Help-seeking/treatment"
    if "advice" in s or "info" in s or "information" in s or "suggest" in s:
        return "Information seeking"
    if "share" in s or "experience" in s or "story" in s or "discussion" in s:
        return "Sharing/community"

    return "No clear action"

df["Cue_group"] = df["Cue to Action"].apply(map_cue_action)
print("Cue_group distribution:")
print(df["Cue_group"].value_counts(dropna=False))

Cue_group distribution:
Cue_group
No clear action           608
Information seeking       105
Help-seeking/treatment     87
Name: count, dtype: int64


In [13]:
# ============================================================
# 9. ROOT CAUSE  → Root_cause_final
#     (adapted to match your clean updated 2 logic)
# ============================================================
candidate_root_cols = [c for c in df.columns if "root" in c.lower() or "label" in c.lower()]
print("Root-cause candidate columns:", candidate_root_cols)

# If this is wrong, manually set ROOT_COL to the correct column (e.g. "Root cause")
ROOT_COL = candidate_root_cols[0]
print("Using root cause column:", ROOT_COL)

df_root = df.dropna(subset=[ROOT_COL]).copy()
df_root[ROOT_COL] = df_root[ROOT_COL].astype(str).str.strip()

print("Root raw value counts:")
print(df_root[ROOT_COL].value_counts(dropna=False))

root_map = {
    "Personality": "Personality",
    "personality": "Personality",

    "Trauma and stress": "Trauma/Stress",
    "Trauma & Stress": "Trauma/Stress",
    "Stress": "Trauma/Stress",
    "Trauma": "Trauma/Stress",

    "Early life": "Early Life",
    "Early life experiences": "Early Life",

    "Drug and alcohol": "Substance Use",
    "Drugs and alcohol": "Substance Use",
    "Substance use": "Substance Use",
    "Alcohol": "Substance Use",
    "Addiction": "Substance Use",
}

def map_root_cause(x):
    x = str(x).strip()
    return root_map.get(x, x)

df_root["Root_cause_clean"] = df_root[ROOT_COL].apply(map_root_cause)
print("Root_cause_clean value counts:")
print(df_root["Root_cause_clean"].value_counts(dropna=False))

min_samples_root = 30
vc_root = df_root["Root_cause_clean"].value_counts()
valid_labels_root = vc_root[vc_root >= min_samples_root].index.tolist()
print("Root cause labels kept:", valid_labels_root)

def merge_rare_root(label):
    if label in valid_labels_root:
        return label
    return "Others"

df_root["Root_cause_final"] = df_root["Root_cause_clean"].apply(merge_rare_root)
print("Root_cause_final distribution:")
print(df_root["Root_cause_final"].value_counts(dropna=False))

Root-cause candidate columns: ['Root cause']
Using root cause column: Root cause
Root raw value counts:
Root cause
Drug and Alcohol     200
Early life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64
Root_cause_clean value counts:
Root_cause_clean
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64
Root cause labels kept: ['Drug and Alcohol', 'Early Life', 'Personality', 'Trauma and Stress']
Root_cause_final distribution:
Root_cause_final
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64


In [14]:
# ============================================================
# MERGE Root_cause_final BACK INTO df
# ============================================================

df = df.merge(
    df_root[["text", "Root_cause_final"]],
    on="text",
    how="left"
)

print("After merging Root_cause_final:")
print(df["Root_cause_final"].value_counts(dropna=False))


After merging Root_cause_final:
Root_cause_final
Trauma and Stress    203
Personality          201
Early Life           200
Drug and Alcohol     200
Name: count, dtype: int64


In [15]:
# ============================================================
# 10. PERCEIVED BENEFITS  → benefit_final
# ============================================================
candidate_benefit_cols = [c for c in df.columns if "benefit" in c.lower()]
print("Benefit candidate columns:", candidate_benefit_cols)

BEN_COL = candidate_benefit_cols[0]  # change manually if needed
print("Using benefit column:", BEN_COL)

df_benefit = df.dropna(subset=[BEN_COL]).copy()
df_benefit[BEN_COL] = df_benefit[BEN_COL].astype(str).str.strip()

print("Benefit raw value counts:")
print(df_benefit[BEN_COL].value_counts(dropna=False))

benefit_map = {
    "support": "Support",
    "finding support": "Support",

    "feeling heard": "Validation",
    "validation": "Validation",
    "feel understood": "Validation",

    "treatment": "Treatment",
    "access to treatment": "Treatment",
    "therapy": "Treatment",

    "not mentioned": "Not Mentioned",
    "none": "Not Mentioned",
}

def map_benefit(x):
    x = str(x).strip().lower()
    return benefit_map.get(x, x.title())

df_benefit["benefit_clean"] = df_benefit[BEN_COL].apply(map_benefit)
print("benefit_clean distribution:")
print(df_benefit["benefit_clean"].value_counts(dropna=False))

min_samples_ben = 30
vc_ben = df_benefit["benefit_clean"].value_counts()
valid_labels_ben = vc_ben[vc_ben >= min_samples_ben].index.tolist()
print("Benefit labels kept:", valid_labels_ben)

def merge_rare_benefits(label):
    if label in valid_labels_ben:
        return label
    return "Others"

df_benefit["benefit_final"] = df_benefit["benefit_clean"].apply(merge_rare_benefits)
print("benefit_final distribution:")
print(df_benefit["benefit_final"].value_counts(dropna=False))

Benefit candidate columns: ['Perceived benefits']
Using benefit column: Perceived benefits
Benefit raw value counts:
Perceived benefits
Not Mentioned                   458
Finding support                 157
Getting access to treatment     124
Improving quality of life        18
Feeling heard and understood     17
Making progress                  16
Feeling relieved                 11
Do not want to bother others      3
Name: count, dtype: int64
benefit_clean distribution:
benefit_clean
Not Mentioned                   458
Support                         157
Getting Access To Treatment     124
Improving Quality Of Life        18
Feeling Heard And Understood     17
Making Progress                  16
Feeling Relieved                 11
Do Not Want To Bother Others      3
Name: count, dtype: int64
Benefit labels kept: ['Not Mentioned', 'Support', 'Getting Access To Treatment']
benefit_final distribution:
benefit_final
Not Mentioned                  458
Support                        157
G

In [16]:
# ============================================================
# MERGE benefit_final BACK INTO df
# ============================================================

df = df.merge(
    df_benefit[["text", "benefit_final"]],
    on="text",
    how="left"
)

print("After merging benefit_final:")
print(df["benefit_final"].value_counts(dropna=False))


After merging benefit_final:
benefit_final
Not Mentioned                  470
Support                        169
Getting Access To Treatment    124
Others                          65
Name: count, dtype: int64


In [17]:
print(df.columns.tolist())


['score', 'selftext', 'subreddit', 'title', 'Root cause', 'Gender', 'Sentiment', 'Perceived severity', 'Perceived benefits', 'Perceived barriers', 'Cue to Action', 'Self-efficacy', 'text', 'severity_merged', 'Self_eff_merged', 'Cue_group', 'Root_cause_final', 'benefit_final']


In [18]:
TASK_COLUMNS = {
    "sentiment": "Sentiment",
    "severity": "severity_merged",
    "self_efficacy_binary": "Self_eff_merged",
    "self_efficacy_4way": "Self-efficacy",
    "cue_to_action": "Cue_group",
    "root_cause": "Root_cause_final",
    "perceived_benefits": "benefit_final"
}


In [19]:
def format_mistral_prompt(text):
    return (
        "You are a mental health analysis assistant.\n"
        "Classify the following text into the correct psychological category.\n\n"
        f"Text: {text}\n"
        "Answer:"
    )


In [20]:
class MistralDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        prompt = format_mistral_prompt(self.texts[idx])

        encoding = self.tokenizer(
            prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }


In [22]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [23]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj",
        "up_proj", "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)


In [24]:
pip install sentencepiece


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
from transformers import AutoTokenizer

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

tokenizer.pad_token = tokenizer.eos_token


In [26]:
results = []

for task_name, label_col in TASK_COLUMNS.items():
    print(f"\n=== Running task: {task_name} ===")

    df_task = df.dropna(subset=[label_col]).copy()

    # Encode labels
    le = LabelEncoder()
    df_task["label_id"] = le.fit_transform(df_task[label_col])

    num_labels = len(le.classes_)
    print("Classes:", list(le.classes_))

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        df_task["text"].tolist(),
        df_task["label_id"].tolist(),
        test_size=0.2,
        stratify=df_task["label_id"],
        random_state=42
    )

    train_dataset = MistralDataset(X_train, y_train, tokenizer)
    eval_dataset = MistralDataset(X_test, y_test, tokenizer)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
    output_dir=f"./mistral_{task_name}",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-4,
    num_train_epochs=5,
    logging_steps=50
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()

    results.append({
        "task": task_name,
        "accuracy": metrics["eval_accuracy"],
        "macro_f1": metrics["eval_macro_f1"],
        "weighted_f1": metrics["eval_weighted_f1"]
    })



=== Running task: sentiment ===
Classes: ['Negative', 'Neutral', 'Positive']


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.3 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 41,955,328 || all params: 7,155,773,440 || trainable%: 0.5863


/tmp/ipykernel_648/3843942945.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


...


=== Running task: perceived_benefits ===
Classes: ['Getting Access To Treatment', 'Not Mentioned', 'Others', 'Support']


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of MistralForSequenceClassification were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.3 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_648/3843942945.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


trainable params: 41,959,424 || all params: 7,155,781,632 || trainable%: 0.5864


Step,Training Loss
50,6.522400
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000


In [27]:
results_df = pd.DataFrame(results)
print("\n=== FINAL MISTRAL-7B QLoRA RESULTS ===")
print(results_df)

results_df.to_excel("mistral_7b_qlora_all_tasks.xlsx", index=False)



=== FINAL MISTRAL-7B QLoRA RESULTS ===
                   task  accuracy  macro_f1  weighted_f1
0             sentiment  0.626506  0.256790     0.482642
1              severity  0.349398  0.103571     0.180938
2  self_efficacy_binary  0.337349  0.252252     0.170194
3    self_efficacy_4way  0.036145  0.017442     0.002522
4         cue_to_action  0.120482  0.071685     0.025910
5            root_cause  0.240964  0.097087     0.093578
6    perceived_benefits  0.150602  0.065445     0.039425
